In [50]:
#!pip install transformers
#!pip install torch torchvision
#!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [73]:
# load libraries
import pandas as pd
import json
import random
from sklearn.metrics import classification_report
from seqeval.metrics import classification_report
from transformers import RobertaTokenizerFast
from transformers import RobertaForTokenClassification
import torch, torchvision
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset
from tqdm import tqdm

In [79]:
# load the annotated data in json format
with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

# initialize tag dictionary
tag_dict = {"O"}

# loop through all sentences
for task in data:
    # loop through all annotations per sentence
    for result in task["annotations"][0]["result"]:
        # check if annoation is of type label
        if result["type"] == "labels":
            # check if there is actually only one label per annotation
            num_labels = len(result["value"]["labels"])
            if num_labels > 1:
                print("More than one label assigned")
            # get the label (only social group, not the sentiment) and add to the dictionary
            label = result["value"]["labels"][0][0:2]
            tag_dict.add(f"B-{label}")
            tag_dict.add(f"I-{label}")

# sort the tag dictionary
tag_list = sorted(tag_dict)

# dictionaries that convert from id to tag and vice versa
tag_to_id = {tag: i for i, tag in enumerate(tag_list)}
id_to_tag = {id: label for label, id in tag_to_id.items()}

In [80]:
# load the tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

# function that creates BIO-tags for text
def tokenize_and_align_labels(text, entities, tag_to_id):

    # tokenize and get offsets
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True)
    # initialize label list with as many "O" labels as there are tokens
    tags = ["O"] * len(encoding.offset_mapping)
    
    # loop through all spans, get start and end position as well as the label
    for ent in entities:
        start, end = ent["start"], ent["end"]
        ent_tag = ent["labels"]
        
        # loop through all tokens in the sentence
        for idx, (token_start, token_end) in enumerate(encoding.offset_mapping):
            if token_start == start:
                tags[idx] = f"B-{ent_tag}"
            if token_start > start and token_end <= end:
                tags[idx] = f"I-{ent_tag}"

    # convert the labels to ids and return
    tag_ids = [tag_to_id.get(label, tag_to_id["O"]) for label in labels]

    return encoding["input_ids"], encoding["attention_mask"], tag_ids

In [84]:
# initialize empty dataset list
dataset = []

# loop through all sentences in the data
for task in data:
    # get the sentence and all annotations
    text = task["data"]["sentence"]
    results = task["annotations"][0]["result"]
    # store annotation spans in a list
    spans = [
        {
            "start": r["value"]["start"],
            "end": r["value"]["end"],
            "labels": r["value"]["labels"][0][0:2]
        }
        for r in results if r["type"] == "labels"
    ]
    # tokenize and get tag ids
    input_ids, attention_mask, tag_ids = tokenize_and_align_labels(text, spans, tag_to_id)

    # get the word ids for all tokens
    encoding = tokenizer(text, truncation=True)
    word_ids = encoding.word_ids() 

    # add everything to the dataset list
    dataset.append({
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "tags": tag_ids,
        "text": text,
        "word_ids": word_ids
    })

In [85]:
# split into training and test dataset
split_idx = int(len(dataset) * 0.75)
train_dataset = dataset[:split_idx]
test_dataset = dataset[split_idx:]

In [86]:
class TokenDataset(Dataset):
    def __init__(self, dataset):
        self.input_ids = [item["input_ids"] for item in dataset]
        self.attention_masks = [item["attention_mask"] for item in dataset]
        self.tags = [item["tags"] for item in dataset]
        self.word_ids = [item["word_ids"] for item in dataset]

        self.max_len = max(len(seq) for seq in self.input_ids)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        # pad sequences
        input_ids = self.input_ids[idx] + [1] * (self.max_len - len(self.input_ids[idx]))
        attention_mask = self.attention_masks[idx] + [0] * (self.max_len - len(self.attention_masks[idx]))
        tags = self.tags[idx] + [-100] * (self.max_len - len(self.tags[idx]))

        # pad word_ids with None (leave as list, not tensor)
        word_ids = self.word_ids[idx] + [None] * (self.max_len - len(self.word_ids[idx]))

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "tags": torch.tensor(tags, dtype=torch.long),
            "word_ids": word_ids
        }

def custom_collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_masks = torch.stack([item["attention_mask"] for item in batch])
    tags = torch.stack([item["tags"] for item in batch])
    word_ids = [item["word_ids"] for item in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "tags": tags,
        "word_ids": word_ids
    }

train_dataset_tensor = TokenDataset(train_dataset)
test_dataset_tensor = TokenDataset(test_dataset)

train_dataloader = DataLoader(
    train_dataset_tensor,
    batch_size=16,
    shuffle=True,
    collate_fn=custom_collate_fn
)

test_dataloader = DataLoader(
    test_dataset_tensor,
    batch_size=16,
    shuffle=False,
    collate_fn=custom_collate_fn
)

In [87]:
# model setup
model = RobertaForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(tag_to_id),
    id2label=id_to_tag,
    label2id=tag_to_id
)

# use gpu if available, otherwise cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# define optimizer, learning rate and the number of epochs
optimizer = AdamW(model.parameters(), lr=1e-5)
epochs = 7

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [62]:
# set the model to training mode
model.train()

# loop through epochs
for epoch in range(epochs):

    # print the epoch number
    print(f"Epoch {epoch + 1}/{epochs}")

    # initialize training loss for the epoch
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc="Training")

    # loop through each batch
    for batch in progress_bar:

        # move all batch data to respective device
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # clear the old gradient
        model.zero_grad()

        # run data through the model and save the outputs
        outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)

        # save the loss and add to the total loss for the epoch
        loss = outputs.loss
        total_loss += loss.item()

        # compute gradients by backpropagation, cap gradients to prevent gradient explosion
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)

        # update the model weights based on the gradient and update the progress bar
        optimizer.step()
        progress_bar.set_postfix(loss=loss.item())

    # get the average training loss per batch and print
    avg_loss = total_loss / len(train_dataloader)
    print(f"Average training loss: {avg_loss:.4f}")


Epoch 1/7


Training: 100%|██████████| 47/47 [01:20<00:00,  1.72s/it, loss=0.0725]


Average training loss: 0.3659
Epoch 2/7


Training: 100%|██████████| 47/47 [01:19<00:00,  1.69s/it, loss=0.145] 


Average training loss: 0.0945
Epoch 3/7


...

Average training loss: 0.0262
Epoch 6/7


Training: 100%|██████████| 47/47 [01:19<00:00,  1.69s/it, loss=0.0471] 


Average training loss: 0.0195
Epoch 7/7


Training: 100%|██████████| 47/47 [01:22<00:00,  1.75s/it, loss=0.00195]

Average training loss: 0.0150


In [64]:
def labels_to_wordlevel_tags(predicted_labels, id_to_tag, word_ids):    
    # intialize dictionary to store all tags assigned to individual words
    word_tags = {}

    # loop through word ids
    for idx, wid in enumerate(word_ids):
        # skip special tokens
        if wid is None:
            continue
        # get the bio-tag assigned to the token
        label = predicted_labels[idx]
        tag = id_to_tag[label][0]
        # add the wid and the assigned token to the dictionary
        if wid not in word_tags:
            word_tags[wid] = []
        word_tags[wid].append(tag)

    # get all unique word ids
    unique_ids = sorted(word_tags.keys())
    
    # initialize list of final labels per word
    final_tags = []

    for wid in sorted(word_tags.keys()):
        # get all unique labels assigned to this word
        unique_labels = set(word_tags[wid])
        if "O" in unique_labels:
            final_tags.append("O")
        elif "B" in unique_labels:
            final_tags.append("B")
        elif "I" in unique_labels:
            final_tags.append("I")
        else:
            final_tags.append("O")

    return final_tags, unique_ids

def extract_spans(word_tags, word_ids):
    
    # empty lists to collect all spans and the current span
    spans = []
    current_span = []

    for wid, label in zip(word_ids, word_tags):
        if label == "B":
            if current_span:
                spans.append(current_span)
            current_span = [wid]
        elif label == "I":
            current_span.append(wid)
        else:
            if current_span:
                spans.append(current_span)
                current_span = []
        
    if current_span:
        spans.append(current_span)

    return spans

def mention_level_evaluation(all_predicted_spans, all_true_spans):

    span_metrics = []
    
    for sentence_preds, sentence_true in zip(all_predicted_spans, all_true_spans):
        pred_sets = [set(p) for p in sentence_preds]
        true_sets = [set(gt) for gt in sentence_true]
        matched_true_idx = set()

        for p_set in pred_sets:
            best_overlap = 0
            best_idx = None
            for i, t_set in enumerate(true_sets):
                overlap = len(p_set & t_set)
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_idx = i
            
            if best_overlap > 0:
                t_set = true_sets[best_idx]
                precision = best_overlap / len(p_set)
                recall = best_overlap / len(t_set)
                f1 = (2*precision*recall)/(precision+recall)
                matched_true_idx.add(best_idx)
            
            else:
                precision, recall, f1 = 0.0, 0.0, 0.0
            span_metrics.append({"precision": precision,
                                 "recall": recall,
                                 "f1": f1})
            
        for i, t_set in enumerate(true_sets):
            if i not in matched_true_idx:
                span_metrics.append({"precision": 0.0,
                                     "recall": 0.0,
                                     "f1": 0.0})
        
    avg_precision = sum(m["precision"] for m in span_metrics) / len(span_metrics)
    avg_recall = sum(m["recall"] for m in span_metrics) / len(span_metrics)
    avg_f1 = sum(m["f1"] for m in span_metrics) / len(span_metrics)

    return {
        "precision": avg_precision,
        "recall": avg_recall,
        "f1": avg_f1
    }

In [70]:
all_predicted_spans = []
all_true_spans = []

# set the model to evaluation mode (no loss calculation)
model.eval()

# proceed without calculating gradients
with torch.no_grad():

    # loop through all batches in the test dataloader
    for batch in test_dataloader:

        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        batch_word_ids = batch["word_ids"]

        # get model outputs, get logits and get the class labels for the max logit
        outputs = model(input_ids=input_ids, attention_mask=attention_masks)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=2)

        # loop through all labels for all sentences in the batch
        for i in range(len(labels)):
            
            # get the true and the predicted label as well as the word ids
            true_seq = labels[i].cpu().numpy()
            pred_seq = predictions[i].cpu().numpy()
            word_ids = batch_word_ids[i]

            word_level_tags, unique_ids = labels_to_wordlevel_tags(pred_seq, id_to_label, word_ids)
            all_predicted_spans.append(extract_spans(word_level_tags, unique_ids))

            word_level_tags, unique_ids = labels_to_wordlevel_tags(true_seq, id_to_label, word_ids)
            all_true_spans.append(extract_spans(word_level_tags, unique_ids))
            
mention_level_evaluation(all_predicted_spans, all_true_spans)

{'precision': 0.7423913043478261,
 'recall': 0.6681465273856578,
 'f1': 0.6746833631344501}

In [72]:
# evaluation at the token level

# set the model to evaluation mode (no loss calculation)
model.eval()

# initialize empty lists for true and predicted labels
true_labels = []
pred_labels = []

# proceed without calculating gradients
with torch.no_grad():

    # loop through all batches in the test dataloader
    for batch in test_dataloader:

        # move all batch data to respective device
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # get model outputs, get logits and get the class labels for the max logit
        outputs = model(input_ids=input_ids, attention_mask=attention_masks)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=2)

        # loop through all labels for all sentences in the batch
        for i in range(len(labels)):
            
            # get the true and the predicted label
            true_seq = labels[i].cpu().numpy()
            pred_seq = predictions[i].cpu().numpy()

            # append true and predicted label only if the true label is not -100 (special token)
            for t, p in zip(true_seq, pred_seq):
                if t != -100:
                    true_labels.append(t)
                    pred_labels.append(p)

# print the classification report
print(classification_report(
    true_labels,
    pred_labels
))

              precision    recall  f1-score   support

           0       0.73      0.77      0.75       155
           1       0.66      0.50      0.57       195
           2       0.99      0.99      0.99      7737

    accuracy                           0.97      8087
   macro avg       0.79      0.75      0.77      8087
weighted avg       0.97      0.97      0.97      8087



In [74]:
# evaluation at the entity level with seqeval
model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_masks)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=2)

        for i in range(len(labels)):
            true_seq = labels[i].cpu().numpy()
            pred_seq = predictions[i].cpu().numpy()

            true_tags = []
            pred_tags = []

            for t, p in zip(true_seq, pred_seq):
                if t != -100:  # ignore padding/special tokens
                    true_tags.append(id_to_label[t])  # convert ID → label string
                    pred_tags.append(id_to_label[p])

            true_labels.append(true_tags)
            pred_labels.append(pred_tags)

print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

          sg       0.62      0.69      0.65       155

   micro avg       0.62      0.69      0.65       155
   macro avg       0.62      0.69      0.65       155
weighted avg       0.62      0.69      0.65       155



In [37]:
# define function for predicting a specific sentence
def predict_sentence(sentence, model, tokenizer, id_to_label, device='cpu'):

    # set model to evaluation mode
    model.eval()

    # tokenize the input sentence, return PyTorch tensors and offset mappings
    encoding = tokenizer(
        sentence,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True
    )

    # extract input ids, attention mask and offset mapping
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    offset_mapping = encoding["offset_mapping"][0]

    # without computing gradients run the input through the trained model
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)[0]

    # get the tokens and all predicted labels for the sentence
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    labels = [id_to_label[pred.item()] for pred in predictions]

    # combine tokens and labels, ignoring special tokens (CLS and end of sequence)
    result = []
    for token, label, (start, end) in zip(tokens, labels, offset_mapping):
        if start == 0 and end == 0:
            continue
        token_text = sentence[start:end]
        result.append((token_text, label))

    return result

In [38]:
# try out a custom sentence
predict_sentence("We are the party that supports the rights of women's children.", model=model, tokenizer=tokenizer, id_to_label=id_to_label)

[('We', 'O'),
 ('are', 'O'),
 ('the', 'O'),
 ('party', 'O'),
 ('that', 'O'),
 ('supports', 'O'),
 ('the', 'O'),
 ('rights', 'O'),
 ('of', 'O'),
 ('women', 'B-sg'),
 ("'s", 'I-sg'),
 ('children', 'I-sg'),
 ('.', 'O')]